In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

print("GPU 사용 여부:", tf.config.list_physical_devices('GPU'))
print("TF 버전:", tf.__version__)

DATA_DIR = '/content/drive/MyDrive/헬스케어 2차/model_hackathon/dataset'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
from PIL import Image, ImageEnhance, ImageFilter
import random

def augment_image(img):
    w, h = img.size
    crop_margin = min(20, w // 6, h // 6)  # 이미지 크기에 맞게 여백 자동 조정
    
    augmentations = [
        lambda x: x.transpose(Image.FLIP_LEFT_RIGHT),
        lambda x: x.rotate(random.uniform(-20, 20), expand=False),
        lambda x: x.rotate(90),
        lambda x: x.rotate(180),
        lambda x: x.rotate(270),
        lambda x: ImageEnhance.Brightness(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Contrast(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: ImageEnhance.Color(x).enhance(random.uniform(0.7, 1.3)),
        lambda x: x.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5))),
        lambda x: x.crop((
            random.randint(0, crop_margin),
            random.randint(0, crop_margin),
            x.width - random.randint(0, crop_margin),
            x.height - random.randint(0, crop_margin)
        )).resize(x.size),
    ]
    return random.choice(augmentations)(img)

TARGET = 300

for cls in sorted(os.listdir(DATA_DIR)):
    cls_path = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_path):
        continue

    files = [f for f in os.listdir(cls_path) if f.lower().endswith('.jpg')]
    current = len(files)
    need = TARGET - current

    if need <= 0:
        print(f"✅ {cls}: {current}장 (증강 불필요)")
        continue

    print(f"증강 중: {cls} ({current}장 → {TARGET}장, {need}장 생성)")
    for i in range(need):
        src_file = random.choice(files)
        src_path = os.path.join(cls_path, src_file)
        img = Image.open(src_path).convert('RGB')
        aug_img = augment_image(img)
        aug_img.save(os.path.join(cls_path, f"aug_{i:04d}.jpg"), quality=90)

print("\n전체 증강 완료!")

In [ ]:
from sklearn.model_selection import train_test_split

all_paths = []
all_labels = []

class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
NUM_CLASSES = len(class_names)

for class_name in class_names:
    class_dir = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(class_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(class_to_idx[class_name])

print(f"총 수집된 이미지: {len(all_paths)}장")
print(f"클래스 수: {NUM_CLASSES}")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.15,
    stratify=all_labels,
    random_state=42
)

print(f"train: {len(train_paths)}장  |  val: {len(val_paths)}장")

In [ ]:
def load_and_augment(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def load_only(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = (train_ds
    .shuffle(len(train_paths))
    .map(load_and_augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = (val_ds
    .map(load_only, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("데이터 파이프라인 완료")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weight_dict = dict(enumerate(class_weights))
print("클래스 가중치 설정 완료")

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S

base_model = EfficientNetV2S(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("모델 구성 완료")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_phase1 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint('best_phase1.keras', save_best_only=True, monitor='val_accuracy')
]

print("=== 1단계: 분류 헤드 학습 ===")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_phase1,
    class_weight=class_weight_dict
)

In [ ]:
base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.8)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.3, patience=4, min_lr=1e-7),
    ModelCheckpoint('best_final.keras', save_best_only=True, monitor='val_accuracy')
]

print("=== 2단계: Fine-tuning ===")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks_phase2,
    class_weight=class_weight_dict
)

In [ ]:
def plot_history(h1, h2):
    acc = h1.history['accuracy'] + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss'] + h2.history['loss']
    val_loss = h1.history['val_loss'] + h2.history['val_loss']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(acc, label='Train Accuracy')
    axes[0].plot(val_acc, label='Val Accuracy')
    axes[0].axvline(x=len(h1.history['accuracy'])-1, color='red',
                    linestyle='--', label='Fine-tune 시작')
    axes[0].set_title('Accuracy')
    axes[0].legend()

    axes[1].plot(loss, label='Train Loss')
    axes[1].plot(val_loss, label='Val Loss')
    axes[1].axvline(x=len(h1.history['loss'])-1, color='red',
                    linestyle='--', label='Fine-tune 시작')
    axes[1].set_title('Loss')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history1, history2)

In [ ]:
from tensorflow.keras.preprocessing import image

idx_to_class = {v: k for k, v in class_to_idx.items()}

def predict_image(img_path, top_k=5):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)[0]
    top_indices = np.argsort(preds)[::-1][:top_k]

    plt.figure(figsize=(5, 5))
    plt.imshow(image.load_img(img_path, target_size=IMG_SIZE))
    plt.axis('off')
    plt.title(f"예측: {idx_to_class[top_indices[0]]} ({preds[top_indices[0]]*100:.1f}%)")
    plt.show()

    print("=" * 40)
    print(f"{'순위':<5} {'클래스':<20} {'확률'}")
    print("-" * 40)
    for rank, idx in enumerate(top_indices, 1):
        print(f"{rank:<5} {idx_to_class[idx]:<20} {preds[idx]*100:.2f}%")
    print("=" * 40)

In [ ]:
from google.colab import files

print("이미지 파일을 업로드하세요:")
uploaded = files.upload()

for filename in uploaded.keys():
    predict_image(filename)